# Longitudinal Data Pipeline

Builds the 60-month longitudinal representation used by the recurrent baseline.

This is the publication copy for the SAGE-Credit repository. The original analysis logic is preserved, machine-specific paths are replaced with portable configuration, and cached outputs are cleared.

> Download the Home Credit Default Risk competition files separately; the dataset is not redistributed here.

In [ ]:
from pathlib import Path
import os

# Portable repository configuration. Override either location with an environment variable.
WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR.parent if WORKING_DIR.name == "notebooks" else WORKING_DIR
DATA_DIR = Path(os.environ.get("HOME_CREDIT_DATA_DIR", PROJECT_ROOT / "data" / "raw")).expanduser().resolve()
ARTIFACT_DIR = Path(os.environ.get("SAGE_CREDIT_ARTIFACT_DIR", PROJECT_ROOT / "artifacts")).expanduser().resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data directory: {DATA_DIR}")
print(f"Artifact directory: {ARTIFACT_DIR}")


In [ ]:
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

application_train = pd.read_csv(DATA_DIR / "application_train.csv")

def clean():
  gc.collect()

In [ ]:
LOOKBACK_MONTHS = 60
keep_app_cols = [
    'SK_ID_CURR', 'TARGET',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION',
    'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE',
    'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'REGION_POPULATION_RELATIVE',
    'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY',
    'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE'
]

app_static = application_train[keep_app_cols].copy()

months = pd.DataFrame({'MONTHS_BALANCE': range(LOOKBACK_MONTHS * -1, 0)})

In [ ]:
app_static = pd.get_dummies(app_static)

for col in app_static.columns:
  if col!='SK_ID_CURR' and col!='TARGET':
    app_static[col] = app_static[col].astype('float32')

del application_train
clean()

In [ ]:
master = app_static.merge(
    months,
    how='cross'
)

master.shape

del app_static, months
clean()

In [ ]:
credit_card_balance = pd.read_csv(DATA_DIR / "credit_card_balance.csv")
LOOKBACK_MONTHS = 60
credit_card_balance=credit_card_balance[credit_card_balance['MONTHS_BALANCE']>= -LOOKBACK_MONTHS].copy()

cc_monthly = credit_card_balance.groupby(['SK_ID_CURR','MONTHS_BALANCE']).agg({
    'SK_ID_PREV': 'nunique',

    'AMT_BALANCE': 'sum',
    'AMT_CREDIT_LIMIT_ACTUAL': 'sum',

    'AMT_DRAWINGS_ATM_CURRENT': 'sum',
    'AMT_DRAWINGS_CURRENT': 'sum',
    'AMT_DRAWINGS_OTHER_CURRENT': 'sum',
    'AMT_DRAWINGS_POS_CURRENT': 'sum',

    'AMT_INST_MIN_REGULARITY': 'sum',

    'AMT_PAYMENT_CURRENT': 'sum',
    'AMT_PAYMENT_TOTAL_CURRENT': 'sum',

    'AMT_RECEIVABLE_PRINCIPAL': 'sum',
    'AMT_RECIVABLE': 'sum',
    'AMT_TOTAL_RECEIVABLE': 'sum',

    'CNT_DRAWINGS_ATM_CURRENT': 'sum',
    'CNT_DRAWINGS_CURRENT': 'sum',
    'CNT_DRAWINGS_OTHER_CURRENT': 'sum',
    'CNT_DRAWINGS_POS_CURRENT': 'sum',

    'CNT_INSTALMENT_MATURE_CUM': 'max',

    'SK_DPD': 'max',
    'SK_DPD_DEF': 'max'
    }).reset_index()

In [ ]:
new_cols=[]

for col in cc_monthly.columns:
  if col=='SK_ID_CURR' or col=='MONTHS_BALANCE':
    new_cols.append(col)
  else:
    new_cols.append('CC_'+col)

cc_monthly.columns = new_cols

In [ ]:
for col in cc_monthly.columns:
  if col!='SK_ID_CURR' and col!='MONTHS_BALANCE':
    cc_monthly[col] = cc_monthly[col].astype('float32')

del credit_card_balance
clean()

In [ ]:
master = master.merge(
    cc_monthly,
    on=['SK_ID_CURR','MONTHS_BALANCE'],
    how='left',
    sort=False
)
print(master.shape)
del cc_monthly
clean()

In [ ]:
POS_CASH_balance = pd.read_csv(DATA_DIR / "POS_CASH_balance.csv")
pos_monthly = POS_CASH_balance.groupby(['SK_ID_CURR','MONTHS_BALANCE']).agg(
    {
        'SK_ID_PREV':'nunique',

        'CNT_INSTALMENT':'sum',
        'CNT_INSTALMENT_FUTURE':'sum',

        'SK_DPD':'max',
        'SK_DPD_DEF':'max'
    }
).reset_index()

In [ ]:
pos_monthly.columns = [
    'SK_ID_CURR',
    'MONTHS_BALANCE',
    'POS_NUM_ACCOUNTS',
    'POS_TOTAL_INSTALMENTS',
    'POS_TOTAL_FUTURE_INSTALMENTS',
    'POS_MAX_DPD',
    'POS_MAX_DPD_DEF'
]

In [ ]:
pos_status = pd.get_dummies(
    POS_CASH_balance[['SK_ID_CURR','MONTHS_BALANCE','NAME_CONTRACT_STATUS']],
    columns=['NAME_CONTRACT_STATUS'],
    prefix=['POS_STATUS']
)

pos_status_monthly = pos_status.groupby(['SK_ID_CURR','MONTHS_BALANCE']).sum().reset_index()

pos_monthly = pos_monthly.merge(
    pos_status_monthly,
    on=['SK_ID_CURR','MONTHS_BALANCE'],
    how='left'
)

In [ ]:
for col in pos_monthly.columns:
  if col!='SK_ID_CURR' and col!='MONTHS_BALANCE':
    pos_monthly[col] = pos_monthly[col].astype('float32')

pos_monthly = pos_monthly[pos_monthly['MONTHS_BALANCE'] >= -LOOKBACK_MONTHS]

del POS_CASH_balance, pos_status, pos_status_monthly
clean()

In [ ]:
master = master.merge(
    pos_monthly,
    on=['SK_ID_CURR','MONTHS_BALANCE'],
    how='left'
)
print(master.shape)
del pos_monthly
clean()

In [ ]:
bureau_balance = pd.read_csv(DATA_DIR / "bureau_balance.csv")
bureau = pd.read_csv(DATA_DIR / "bureau.csv")

In [ ]:
bureau_lookup = bureau[['SK_ID_BUREAU','SK_ID_CURR']].copy()

merged_bureau = bureau_balance.merge(
    bureau_lookup,
    on='SK_ID_BUREAU',
    how='left'
)

merged_bureau.head()

In [ ]:
bb_dummies = pd.get_dummies(
    merged_bureau[['SK_ID_CURR','MONTHS_BALANCE','STATUS']],
    columns=['STATUS'],
    prefix='BB_STATUS'
)

In [ ]:
bb_monthly = (bb_dummies.groupby(['SK_ID_CURR','MONTHS_BALANCE']).sum().reset_index())

In [ ]:
for col in bb_monthly.columns:
  if col!='SK_ID_CURR' and col!='MONTHS_BALANCE':
    bb_monthly[col] = bb_monthly[col].astype('float32')

bb_monthly = bb_monthly[bb_monthly['MONTHS_BALANCE'] >= -LOOKBACK_MONTHS]

In [ ]:
del bureau, bureau_balance, bureau_lookup, merged_bureau, bb_dummies
clean()

In [ ]:
master = master.merge(
    bb_monthly,
    on=['SK_ID_CURR','MONTHS_BALANCE'],
    how='left'
)
print(master.shape)
del bb_monthly
clean()

In [ ]:
cc_cols = []
for col in master.columns:
  if (col.startswith('CC_')):
    cc_cols.append(col)

master[cc_cols] = master[cc_cols].fillna(0)

pos_cols = []
for col in master.columns:
  if col.startswith('POS_'):
    pos_cols.append(col)

master[pos_cols] = master[pos_cols].fillna(0)

bb_cols = []
for col in master.columns:
  if col.startswith('BB_'):
    bb_cols.append(col)

master[bb_cols] = master[bb_cols].fillna(0)

del bb_cols, cc_cols, pos_cols
clean()

In [ ]:
app_nan_cols = [
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "AMT_GOODS_PRICE",
    "AMT_ANNUITY",
    "CNT_FAM_MEMBERS",
    "DAYS_LAST_PHONE_CHANGE"
]

for col in app_nan_cols:
    master[col] = master[col].fillna(master[col].median())

del app_nan_cols
clean()

In [ ]:
master['TARGET'].value_counts()
print("Total NaNs:", master.isna().sum().sum())

In [ ]:
master.to_pickle(ARTIFACT_DIR / f"home_credit_master_{LOOKBACK_MONTHS}m_REAL_NOW.pkl")

# **ALL PLOTS**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

master = pd.read_pickle(ARTIFACT_DIR / "home_credit_master_60m_REAL_NOW.pkl")

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 140,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25
})

In [ ]:
candidate_cols = [
    "CC_AMT_BALANCE",
    "CC_AMT_CREDIT_LIMIT_ACTUAL",
    "CC_AMT_DRAWINGS_CURRENT",
    "CC_AMT_PAYMENT_CURRENT",
    "INST_AMT_PAYMENT",
    "INST_AMT_INSTALMENT",
    "POS_CNT_INSTALMENT_FUTURE",
    "POS_SK_DPD",
    "BB_AMT_CREDIT_SUM_DEBT",
    "BB_AMT_CREDIT_SUM"
]

temporal_cols = [c for c in candidate_cols if c in master.columns]

print("Using columns:")
print(temporal_cols)

fig, axes = plt.subplots(
    nrows=len(temporal_cols),
    ncols=1,
    figsize=(10, 3 * len(temporal_cols)),
    sharex=True
)

if len(temporal_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, temporal_cols):
    temp = (
        master
        .groupby(["TARGET", "MONTHS_BALANCE"])[col]
        .mean()
        .reset_index()
    )

    for target, label in [(0, "Non-default"), (1, "Default")]:
        data = temp[temp["TARGET"] == target]
        ax.plot(
            data["MONTHS_BALANCE"],
            data[col],
            linewidth=2,
            label=label
        )

    nice_names = {
    "CC_AMT_BALANCE": "Credit Card Balance",
    "CC_AMT_CREDIT_LIMIT_ACTUAL": "Credit Limit",
    "CC_AMT_DRAWINGS_CURRENT": "Current Drawings",
    "CC_AMT_PAYMENT_CURRENT": "Current Payment",
    "INST_AMT_PAYMENT": "Installment Payment",
    "INST_AMT_INSTALMENT": "Scheduled Installment",
    "POS_CNT_INSTALMENT_FUTURE": "Remaining Installments",
    "POS_SK_DPD": "Days Past Due",
    "BB_AMT_CREDIT_SUM_DEBT": "Bureau Debt",
    "BB_AMT_CREDIT_SUM": "Bureau Credit"
    }

    ax.set_title(nice_names[col])
    ax.set_ylabel("Mean value")
    ax.grid(alpha=0.25)
    ax.legend()

axes[-1].set_xlabel("Months Before Current Application")
axes[-1].invert_xaxis()

plt.suptitle("Average Temporal Feature Trajectories by Target", y=1.01)
plt.tight_layout()
plt.show()